In [1]:
from rsm3d.data_io import RSMDataLoader
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/nsls2/data/staff/xyang4/data_cs/isr_20251020/FeTe_GeTe_101525'
setup_file = './exp_setup.yaml'
tiff_dir  = '/nsls2/data/staff/xyang4/data_cs/isr_20251020/Oct_22_2025'
# tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'  
out_vtr   = '/nsls2/data/staff/xyang4/data_cs/isr_20251020/rsm_hkl_20251020.vtr'    # output file
scan_list = (65,) 

/nsls2/users/xyang4/pyprojects/pyisr/rsm3d/data_io.py:86: SyntaxWarning: invalid escape sequence '\d'
  scan_number and data_number. Defaults to r"^[^_]+_[^_]+_(\d{3})_(\d{3})_.*\\.tiff$".


In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=scan_list,
    process_hklscan_only=True,
)

In [3]:
builder = RSMBuilder(loader, 
                    sample_axes = ('z-', 'y+', 'x+'),
                    detector_axes = ('z-',),
                    ub_includes_2pi=True)
Q_samp, hkl, intensity = builder.compute_full()

TypeError: RSMBuilder.__init__() got an unexpected keyword argument 'sample_axes'

In [ ]:
grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(100, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

In [ ]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [ ]:
from rsm3d.data_io import write_rsm_volume_to_vtr, write_rsm_volume_to_vtk
rsm = grid
edges = (xax, yax, zax)
filename = out_vtr
write_rsm_volume_to_vtk(rsm, edges, filename.replace('.vtr', '.vtk'))
write_rsm_volume_to_vtr(rsm, edges, filename, binary=False, compress=True)